### 1) Imports y carga de datos

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

In [6]:
# Cargar dataset
df = pd.read_csv("../data/adaptive_comfort_dataset.csv")

print("Total filas:", len(df))
df.head()

Total filas: 300


,building_id,neutral_temp,records,cooling_type,region,t_out_mean
0,1,22.585738,170,mixed mode,oceania,15.296857
1,2,22.058339,83,air conditioned,oceania,13.995833
2,3,23.142187,85,air conditioned,americas,0.583480
3,4,23.642083,137,mixed mode,oceania,19.284220
4,5,22.071788,128,air conditioned,americas,9.048210


### 2) Función base para entrenar

In [7]:
def run_experiment(df):
    """
    Ejecuta entrenamiento y evaluación con:
    - t_out_mean
    - cooling_type (one-hot)
    """

    encoder = OneHotEncoder(sparse_output=False)

    X_cat = encoder.fit_transform(df[["cooling_type"]])
    X_num = df[["t_out_mean"]].values

    X = np.hstack([X_num, X_cat]).astype(np.float32)
    y = df["neutral_temp"].values.astype(np.float32)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(n_estimators=15, max_depth=4, random_state=42)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    return rmse, r2, len(df)

### 3) Experimentos
- Experimento 1: records > 50
- Experimento 2: records > 50 + air conditioned
- Experimento 3: records > 50 + air conditioned + Americas

In [8]:
df_exp1 = df[df["records"] > 50].copy()

rmse1, r21, n1 = run_experiment(df_exp1)

print("EXPERIMENTO 1")
print("Filas:", n1)
print("RMSE:", rmse1)
print("R2:", r21)

EXPERIMENTO 1
Filas: 191
RMSE: 1.6843911967224137
R2: 0.3885708129352389


In [ ]:
df_exp2 = df[(df["records"] > 50) & (df["cooling_type"] == "air conditioned")].copy()

rmse2, r22, n2 = run_experiment(df_exp2)

print("\nEXPERIMENTO 2")
print("Filas:", n2)
print("RMSE:", rmse2)
print("R2:", r22)


EXPERIMENTO 2
Filas: 96
RMSE: 0.9132812043800409
R2: -0.7355245085047526


In [ ]:
df_exp3 = df[
    (df["records"] > 50)
    & (df["cooling_type"] == "air conditioned")
    & (df["region"] == "americas")
].copy()

rmse3, r23, n3 = run_experiment(df_exp3)

print("\nEXPERIMENTO 3")
print("Filas:", n3)
print("RMSE:", rmse3)
print("R2:", r23)


EXPERIMENTO 3
Filas: 25
RMSE: 0.6811986859666642
R2: -2.799389222088099


In [ ]:
resultados = pd.DataFrame(
    {
        "Experimento": ["Records > 50", "Air Conditioned", "Air + Americas"],
        "Filas": [n1, n2, n3],
        "RMSE": [rmse1, rmse2, rmse3],
        "R2": [r21, r22, r23],
    }
)

resultados

,Experimento,Filas,RMSE,R2
0,Records > 50,191,1.684391,0.388571
1,Air Conditioned,96,0.913281,-0.735525
2,Air + Americas,25,0.681199,-2.799389
